# For submission purposes only cannot be run without active surreal server + data ingestion

In [1]:
import sys, json
from pathlib import Path

# Make sure project root is on path
sys.path.insert(0, str(Path("..").resolve()))

In [2]:
import pandas as pd

In [3]:
from ingestion.surreal_client import SurrealClient

In [4]:
client = SurrealClient()

In [5]:
def format_query_to_dataframe(query_result):
    if not query_result:
        return "No results found."
    df = pd.DataFrame(query_result)
    if 'id' in df.columns:
        df['id'] = df['id'].astype(str).str.split(':').str[-1]
    return df

def run_and_show(query):
    result = client.http_query(query, False)
    return format_query_to_dataframe(result[0]['result'])

In [6]:
#Retrieve the full names of students who have never attempted any CodeWorkout problems.
run_and_show("""
    SELECT name AS full_name
        FROM student
        WHERE id NOT IN (SELECT VALUE student_id FROM submission);
    """)

[{"result":[{"full_name":"McConney, Brendon"},{"full_name":"Pang, Adam"},{"full_name":"Ghimire, Sami"},{"full_name":"Dukes, Kyrin"},{"full_name":"Gilbert, Andrew"},{"full_name":"Thomas, Aidan"},{"full_name":"Kabir, Ari"},{"full_name":"Conner, Gavin"},{"full_name":"Brown, Demetrius"},{"full_name":"Wegener, Ryan"},{"full_name":"Rich, Connor"},{"full_name":"Singley, Maurice"},{"full_name":"Adigwe, Precious"},{"full_name":"Ali Khan, Mujtaba"},{"full_name":"Anuonye, Martin"},{"full_name":"Asunramu, Oluwatomisin"},{"full_name":"Bachelder, Matthew"},{"full_name":"Benbow, Anna"},{"full_name":"Bentley, Cameron"},{"full_name":"Bruce, Darryl"},{"full_name":"Brummer, Greyson"},{"full_name":"Cancio-Fitzgerald, Mateo"},{"full_name":"Crosby, Ryan"},{"full_name":"Davidson, Emma"},{"full_name":"Diakite, Lansina"},{"full_name":"Dickerson, Summer"},{"full_name":"Dishman, Joe"},{"full_name":"Feldman, Ben"},{"full_name":"Flowers, Josiah"},{"full_name":"Fonseca-Folden, Jax"},{"full_name":"Fowler, Dalton"},{

,full_name
0,"McConney, Brendon"
1,"Pang, Adam"
2,"Ghimire, Sami"
3,"Dukes, Kyrin"
4,"Gilbert, Andrew"
...,...
71,"Vallapalli, Soumith"
72,"Colley, Justin"
73,"Herrera, Natalia"
74,"Luongo, Will"


In [7]:
#Retrieve the full names of students who have the highest total number of submission attempts.
run_and_show("""
    SELECT student_id,type::record('student',student_id).name as full_name,total_attempts
        FROM (SELECT student_id,
        count() AS total_attempts
        FROM submission
        GROUP BY student_id
        ORDER BY total_attempts DESC);
    """)

[{"result":[{"full_name":"Diakite, Lansina","student_id":"77716","total_attempts":3695},{"full_name":"Martinez, Hernan","student_id":"77737","total_attempts":3645},{"full_name":"Yang, Angela","student_id":"77768","total_attempts":3615},{"full_name":"Jimenez, Edwin","student_id":"77733","total_attempts":3519},{"full_name":null,"student_id":"77744","total_attempts":3110},{"full_name":"Jones, Elijah","student_id":"77734","total_attempts":2815},{"full_name":"Nariani, Disha","student_id":"77743","total_attempts":2803},{"full_name":"Benbow, Anna","student_id":"77707","total_attempts":2781},{"full_name":"Fonseca-Folden, Jax","student_id":"77722","total_attempts":2712},{"full_name":"Stegall, Julian","student_id":"77760","total_attempts":2564},{"full_name":"Lai, Kristen","student_id":"77735","total_attempts":2554},{"full_name":"Taylor, Benjamin","student_id":"77762","total_attempts":2474},{"full_name":"Adigwe, Precious","student_id":"77702","total_attempts":2356},{"full_name":"Goliber, Kialey",

,full_name,student_id,total_attempts
0,"Diakite, Lansina",77716,3695
1,"Martinez, Hernan",77737,3645
2,"Yang, Angela",77768,3615
3,"Jimenez, Edwin",77733,3519
4,NaN,77744,3110
...,...,...,...
82,NaN,69461,108
83,NaN,77750,108
84,NaN,77740,106
85,NaN,77719,74


In the above query names are NaN the student id's not appearing in original student mapping file

In [8]:
#Given a Problem ID and a Student SIS Login ID, retrieve the chronological sequence of actions 
# along with their statuses from the student's CodeWorkout interactions.
sis_id = "abenbow1"
problem_id = "915"
run_and_show(f"""
    SELECT server_timestamp, event_type, result, compile_message_type, compile_message
            FROM submission
            WHERE problem_id = '{problem_id}' AND '{sis_id}' IN ->submitted_by->student.sis_id
            ORDER BY server_timestamp ASC;
    """)

[{"result":[{"compile_message":"","compile_message_type":"","event_type":"Run.Program","result":"","server_timestamp":"2024-09-22T19:43:46Z"},{"compile_message":"","compile_message_type":"","event_type":"Compile","result":"Error","server_timestamp":"2024-09-22T19:43:46Z"},{"compile_message":"line 4: error: cannot find symbol: method len()","compile_message_type":"SyntaxError","event_type":"Compile.Error","result":"","server_timestamp":"2024-09-22T19:43:46Z"},{"compile_message":"","compile_message_type":"","event_type":"Compile","result":"Error","server_timestamp":"2024-09-30T15:44:41Z"},{"compile_message":"line 36: error: missing return statement","compile_message_type":"SyntaxError","event_type":"Compile.Error","result":"","server_timestamp":"2024-09-30T15:44:41Z"},{"compile_message":"","compile_message_type":"","event_type":"Run.Program","result":"","server_timestamp":"2024-09-30T15:44:41Z"},{"compile_message":"","compile_message_type":"","event_type":"Run.Program","result":"","serve

,compile_message,compile_message_type,event_type,result,server_timestamp
0,,,Run.Program,,2024-09-22T19:43:46Z
1,,,Compile,Error,2024-09-22T19:43:46Z
2,line 4: error: cannot find symbol: method len(),SyntaxError,Compile.Error,,2024-09-22T19:43:46Z
3,,,Compile,Error,2024-09-30T15:44:41Z
4,line 36: error: missing return statement,SyntaxError,Compile.Error,,2024-09-30T15:44:41Z
5,,,Run.Program,,2024-09-30T15:44:41Z
6,,,Run.Program,,2024-09-30T15:44:47Z
7,line 18: error: cannot find symbol: variable i,SyntaxError,Compile.Error,,2024-09-30T15:44:47Z
8,,,Compile,Error,2024-09-30T15:44:47Z
9,line 4: error: cannot find symbol: variable len,SyntaxError,Compile.Error,,2024-09-30T15:44:50Z


In [9]:
#Given a Problem ID, retrieve all learning concepts assessed by that problem
problem_id = "1397"
run_and_show(f"""
    SELECT VALUE concepts
        FROM cw_problem
        WHERE id = cw_problem:`{problem_id}`;
    """)

[{"result":[["SLL Traversal/Search","Algorithm Analysis","Stack Conceptal View","Linked List","O Notation","SLL Removing","SLL Insertion","Stack","Stack Linked List Implementation","Singly Linked List"]],"status":"OK","time":"134.3µs","type":null}]


,0,1,2,3,4,5,6,7,8,9
0,SLL Traversal/Search,Algorithm Analysis,Stack Conceptal View,Linked List,O Notation,SLL Removing,SLL Insertion,Stack,Stack Linked List Implementation,Singly Linked List
